# Diferenciación Automática con torch.autograd

Al entrenar redes neuronales, el algoritmo más utilizado es la **propagación hacia atrás (backpropagation)**. En este algoritmo, los parámetros (pesos del modelo) se ajustan de acuerdo con el **gradiente de la función de pérdida** con respecto al parámetro dado.

Para calcular esos gradientes, PyTorch tiene un motor de diferenciación integrado llamado **`torch.autograd`**. Soporta el cálculo automático de gradientes para cualquier grafo computacional.

## ¿Qué es la diferenciación automática?

La diferenciación automática (automatic differentiation, AD) es una técnica para calcular derivadas de funciones expresadas como código de computadora. Es fundamental para:
- **Entrenar redes neuronales**: ajustar pesos mediante descenso de gradiente
- **Optimización**: encontrar mínimos/máximos de funciones
- **Backpropagation**: calcular cómo cambia la pérdida con respecto a cada parámetro

## Ejemplo: Red neuronal simple

Consideremos la red neuronal más simple de una capa, con entrada `x`, parámetros `w` y `b`, y alguna función de pérdida. Se puede definir en PyTorch de la siguiente manera:

In [ ]:
import torch

x = torch.ones(5)  # tensor de entrada
y = torch.zeros(3)  # salida esperada
w = torch.randn(5, 3, requires_grad=True)
b = torch.randn(3, requires_grad=True)
z = torch.matmul(x, w)+b
loss = torch.nn.functional.binary_cross_entropy_with_logits(z, y)

## Tensores, Funciones y Grafo Computacional

Este código define el siguiente **grafo computacional**:

```
x ──┐
    ├─→ matmul ──→ z (+ b) ──→ loss (binary_cross_entropy)
w ──┘                ↑
                     b
```

En esta red, `w` y `b` son **parámetros** que necesitamos optimizar. Por lo tanto, necesitamos poder calcular los gradientes de la función de pérdida con respecto a esas variables. Para hacerlo, establecemos la propiedad `requires_grad` de esos tensores.

### ¿Qué es requires_grad?

- **`requires_grad=True`**: Indica a PyTorch que debe rastrear todas las operaciones en este tensor
- **Automático**: PyTorch construye automáticamente el grafo computacional
- **Eficiente**: Solo rastrea lo necesario para calcular gradientes

**Nota:** Puedes establecer el valor de `requires_grad` al crear un tensor, o más tarde usando el método `x.requires_grad_(True)`.

### Funciones y grad_fn

Una función que aplicamos a tensores para construir el grafo computacional es en realidad un objeto de la clase **`Function`**. Este objeto sabe cómo:
1. **Calcular la función en la dirección hacia adelante (forward)**
2. **Calcular su derivada durante el paso de propagación hacia atrás (backward)**

Una referencia a la función de propagación hacia atrás se almacena en la propiedad `grad_fn` de un tensor.

In [ ]:
print(f"Gradient function for z = {z.grad_fn}")
print(f"Gradient function for loss = {loss.grad_fn}")

## Calcular Gradientes

Para optimizar los pesos de los parámetros en la red neuronal, necesitamos calcular las derivadas de nuestra función de pérdida con respecto a los parámetros, es decir, necesitamos $\frac{\partial loss}{\partial w}$ y $\frac{\partial loss}{\partial b}$ bajo algunos valores fijos de `x` e `y`.

Para calcular esas derivadas:
1. Llamamos a `loss.backward()`
2. Recuperamos los valores de `w.grad` y `b.grad`

### ¿Qué hace backward()?

- **Calcula gradientes**: Aplica la regla de la cadena automáticamente
- **Propaga hacia atrás**: Desde la salida hasta las hojas del grafo
- **Acumula en .grad**: Almacena los gradientes en el atributo `.grad` de cada tensor

In [ ]:
loss.backward()
print(w.grad)
print(b.grad)

**Notas importantes:**

1. Solo podemos obtener las propiedades `grad` para los **nodos hoja** del grafo computacional, que tienen la propiedad `requires_grad` establecida en `True`. Para todos los demás nodos en nuestro grafo, los gradientes no estarán disponibles.

2. Solo podemos realizar cálculos de gradiente usando `backward` **una vez** en un grafo dado, por razones de rendimiento. Si necesitamos hacer varias llamadas backward en el mismo grafo, necesitamos pasar `retain_graph=True` a la llamada backward.

### Conceptos clave:

- **Nodos hoja**: Tensores de entrada que tienen `requires_grad=True`
- **Nodos intermedios**: Resultados de operaciones, no almacenan gradientes
- **Raíz del grafo**: La pérdida final desde donde comienza backward

## Deshabilitar el Rastreo de Gradientes

Por defecto, todos los tensores con `requires_grad=True` están rastreando su historial computacional y soportan el cálculo de gradientes. Sin embargo, hay casos en los que no necesitamos hacer eso, por ejemplo:
- Cuando hemos **entrenado el modelo** y solo queremos aplicarlo a algunos datos de entrada
- Solo queremos hacer **cálculos hacia adelante** a través de la red
- **Inferencia**: no necesitamos calcular gradientes

Podemos detener el rastreo de cálculos rodeando nuestro código de cálculo con el bloque **`torch.no_grad()`**:

### Ventajas de deshabilitar gradientes:

- **Menor uso de memoria**: No se almacena el historial computacional
- **Mayor velocidad**: Las operaciones son más rápidas sin rastreo
- **Parámetros congelados**: Útil para fine-tuning de modelos

In [ ]:
z = torch.matmul(x, w)+b
print(z.requires_grad)

with torch.no_grad():
    z = torch.matmul(x, w)+b
print(z.requires_grad)

### Método alternativo: detach()

Otra forma de lograr el mismo resultado es usar el método **`detach()`** en el tensor. Este método crea un nuevo tensor que comparte los mismos datos pero no tiene conexión con el grafo computacional.

In [ ]:
z = torch.matmul(x, w)+b
z_det = z.detach()
print(z_det.requires_grad)

### Razones para deshabilitar el rastreo de gradientes:

1. **Marcar parámetros congelados**: Para usar algunos parámetros en tu red neuronal como **parámetros congelados**. Este es un escenario común para el fine-tuning de modelos preentrenados.

2. **Acelerar cálculos**: Cuando solo estás haciendo el **pase hacia adelante**, porque los cálculos en tensores que no rastrean gradientes serían más eficientes.

3. **Ahorro de memoria**: No almacenar el historial computacional reduce significativamente el uso de memoria.

## Más sobre Grafos Computacionales

Conceptualmente, **autograd** mantiene un registro de datos (tensores) y todas las operaciones ejecutadas (junto con los nuevos tensores resultantes) en un **grafo acíclico dirigido (DAG)** compuesto por objetos `Function`. 

En este DAG:
- Las **hojas** son los tensores de entrada
- Las **raíces** son los tensores de salida

Al rastrear este grafo desde las raíces hasta las hojas, puedes calcular automáticamente los gradientes usando la **regla de la cadena**.

### En un pase hacia adelante, autograd hace dos cosas simultáneamente:

1. **Ejecuta la operación solicitada** para calcular un tensor resultante
2. **Mantiene la función de gradiente** de la operación en el DAG

### El pase hacia atrás comienza cuando se llama `.backward()` en la raíz del DAG:

1. **Calcula los gradientes** desde cada `.grad_fn`
2. **Los acumula** en el atributo `.grad` del tensor respectivo
3. **Usando la regla de la cadena**, propaga todo el camino hasta los tensores hoja

### DAGs son dinámicos en PyTorch

**Importante:** El grafo se recrea desde cero después de cada llamada a `.backward()`. autograd comienza a poblar un nuevo grafo. Esto es exactamente lo que te permite usar declaraciones de control de flujo en tu modelo; puedes cambiar la forma, tamaño y operaciones en cada iteración si es necesario.

## Lectura Opcional: Gradientes de Tensores y Productos Jacobianos

En muchos casos, tenemos una función de pérdida escalar y necesitamos calcular el gradiente con respecto a algunos parámetros. Sin embargo, hay casos en los que la función de salida es un tensor arbitrario. En este caso, PyTorch te permite calcular el llamado **producto Jacobiano**, y no el gradiente real.

### Matriz Jacobiana

Para una función vectorial $\vec{y} = f(\vec{x})$, donde $\vec{x} = \langle x_1, \ldots, x_n \rangle$ y $\vec{y} = \langle y_1, \ldots, y_m \rangle$, un gradiente de $\vec{y}$ con respecto a $\vec{x}$ está dado por la matriz Jacobiana:

$$J = \begin{pmatrix}
\frac{\partial y_1}{\partial x_1} & \cdots & \frac{\partial y_1}{\partial x_n} \\
\vdots & \ddots & \vdots \\
\frac{\partial y_m}{\partial x_1} & \cdots & \frac{\partial y_m}{\partial x_n}
\end{pmatrix}$$

### Producto Jacobiano

En lugar de calcular la matriz Jacobiana en sí, PyTorch te permite calcular el **producto Jacobiano** $v^T \cdot J$ para un vector de entrada dado $v = (v_1 \ldots v_m)$. 

Esto se logra llamando a `backward` con $v$ como argumento. El tamaño de $v$ debe ser el mismo que el tamaño del tensor original, con respecto al cual queremos calcular el producto.

In [ ]:
inp = torch.eye(4, 5, requires_grad=True)
out = (inp+1).pow(2).t()
out.backward(torch.ones_like(out), retain_graph=True)
print(f"First call\n{inp.grad}")
out.backward(torch.ones_like(out), retain_graph=True)
print(f"\nSecond call\n{inp.grad}")
inp.grad.zero_()
out.backward(torch.ones_like(out), retain_graph=True)
print(f"\nCall after zeroing gradients\n{inp.grad}")

### ¿Qué observamos en el ejemplo anterior?

Nota que cuando llamamos `backward` por segunda vez con el mismo argumento, el valor del gradiente es diferente. Esto sucede porque al hacer propagación hacia atrás, PyTorch **acumula los gradientes**, es decir, el valor de los gradientes calculados se **suma** a la propiedad `grad` de todos los nodos hoja del grafo computacional. 

Si quieres calcular los gradientes apropiados, necesitas **poner a cero** la propiedad `grad` antes. En el entrenamiento de la vida real, un **optimizador** nos ayuda a hacer esto.

### Puntos clave sobre acumulación de gradientes:

1. **Primera llamada**: Los gradientes se calculan y almacenan en `.grad`
2. **Segunda llamada**: Los nuevos gradientes se **suman** a los existentes
3. **Después de zero_()**: Los gradientes se resetean y vuelven a calcularse desde cero

### Uso típico en entrenamiento:

```python
optimizer.zero_grad()  # Resetear gradientes
loss.backward()        # Calcular gradientes
optimizer.step()       # Actualizar parámetros
```

**Nota:** Anteriormente llamábamos a la función `backward()` sin parámetros. Esto es esencialmente equivalente a llamar `backward(torch.tensor(1.0))`, que es una forma útil de calcular los gradientes en caso de una función de valor escalar, como la pérdida durante el entrenamiento de redes neuronales.